# [LOG04 미션 07] AI와 함께 물자 부족 상황의 판단 기록과 점검표 만들기 — 학생 실습용 (Student)

- **산출물 경로**: `rubric/log04_evaluation_rubric.md`
- **실습 환경**: Python 3.12 표준 라이브러리 (`pathlib`, `csv`) — 별도 라이브러리 설치 및 인터넷 연결 불필요

> **실습·보안 안내**: 제공된 교육용 가상자료만 사용하며 실제 조직명·물자 제원·재고·개인정보는 입력하지 마세요. 점수나 등급으로 승인을 자동 결정하지 않으며, 최종 판단은 반드시 사람이 내립니다.

---

## 미션 7. AI와 함께 물자 부족 상황의 판단 기록과 점검표 만들기

### 목표

이번 미션에서는 교육장에서 제공하는 생성형 AI와 함께 「물자 부족 상황의 판단 기록과 점검표」 초안을 만듭니다.

운송이 늦어지고 사용량이 바뀌면 기존 문서의 판단도 다시 확인해야 합니다.

상황별 기록을 남기고 같은 기준으로 점검해 문서가 어떤 변화에 약한지 알아봅니다.

준비할 자료는 다음과 같습니다: 상황 카드 T1~T6, 미션 5 절차, 5영역 평가 틀. AI가 제안한 초안을 받은 뒤 사실과 숫자, 빠진 항목, 과장된 표현을 확인하고 직접 고쳐 씁니다.

끝나면 완성한 문서와 자료 대조 기록, 사용한 프롬프트, 고친 이유가 남습니다.

문장마다 근거를 남겨 다음 미션에서도 같은 자료를 찾아 쓸 수 있게 합니다.

수량은 단위와 기간을 함께 적고, 계산 결과는 계산기로 다시 확인합니다.

자료에 없는 값은 추측하지 않고 미확인으로 남깁니다.

다른 담당자가 문장과 표를 읽고 같은 자료를 찾을 수 있게 근거를 붙입니다.

### 상황과 자료

운송 경로 차단과 물자 80% 소진 등 카드 T1~T6이 순서대로 제시됩니다.

A팀은 매번 우선순위를 다시 적고 B팀은 근거를 검토합니다.

**준비할 자료**: 상황 카드 T1~T6, 미션 5 절차, 5영역 평가 틀

**뜻 알아두기**: 루브릭은 기록에서 확인할 행동을 등급별로 적은 점검표입니다.

같은 점수의 영역 수와 전체 영역 수를 함께 적어 일치율을 구합니다.

사전열람·일시정지·목표시간 초과 여부도 기록합니다.


In [ ]:
# [환경 준비] 미션 7 실습 데이터셋 및 빈 산출물 서식 생성 (기존 파일은 보존)
from pathlib import Path

payload = {'02_데이터셋/거점_보유소모_대장.csv': "[교육용가상자료] 거점 보유·소모 대장 — 가상 조직 A팀 3개 거점의 D-3일~D일 기록. 수량 단위는 품목별(발·세트·묶음). 실제 재고·제원과 무관하다. 첫 줄은 안내문, 둘째 줄부터 머리글.\n거점,품목,단위,D-3일보유량,D-2일소모,D-1일소모,D일소모,입고량_D-2~D,D일보유량_보고,일평균소모량_보고,리드타임_일,보고출처,보고시각,비고\n북곡,AM-1,발,42,6,6,6,0,24,6,3,창고 실사+서명,D일 17:00,D-3일 소모는 3발(이전 수준)\n북곡,DP-2,세트,60,5,5,5,0,40,5,2,불출 대장 집계,D일 17:00,파손 손실 기록란 비어 있음\n북곡,PG-3,묶음,150,10,10,10,0,120,10,1,창고 실사,D일 17:00,\n서림,AM-1,발,27,3,3,3,0,18,3,3,담당자 A 대장 집계,D일 18:00,담당자 B는 소모 5/일로 보고(K4)\n서림,DP-2,세트,24,6,6,0,0,12,4,2,불출 대장 집계,D일 18:00,D일 소모는 18시 집계 전(0으로 기재). 파손 손실과 정상 소모가 한 칸에 합산\n서림,PG-3,묶음,75,15,15,15,0,30,15,1,담당자 구두,D일 18:00,수량 '30 정도'의 어림값\n해솔,AM-1,발,20,4,4,4,0,8,4,3,전일 보고서 인용,D-1일 18:00,전일 인용은 D-1일 18시 출처 시각이다. D일 19시 대장 담당자가 D일 소모 4를 계산용 추정 기입하고 보유량 8을 산출했다. D일 실사값 아님. 현재량 확인 후 사용\n해솔,DP-2,세트,26,2,2,2,0,20,2,2,창고 실사,D일 16:00,\n해솔,PG-3,묶음,78,6,6,6,0,60,6,1,창고 실사,D일 16:00,야간 소모량 미반영\n", '02_데이터셋/재보급_요청대장.csv': "[교육용가상자료] 재보급 요청대장 — D일 가상 조직 A팀이 지원팀에 올린 요청. '요청서기재_보유일수'는 요청 거점이 스스로 적은 값이며 대장 재계산과 다를 수 있다. 첫 줄은 안내문, 둘째 줄부터 머리글.\n요청ID,거점,품목,요청수량,요청시각,요청사유,요청서기재_보유일수,근거유형,수송로상태,도착예정,비고\nQ4,해솔,AM-1,10,D일 07:30,보유일수 2일로 리드타임 3일보다 짧음,2.0,전일 보고서 인용,양호,D+3일,R3와 같은 자료\nQ1,북곡,AM-1,12,D일 09:00,현장 대응 빈도 증가로 소모 6발/일,4.0,창고 실사+서명,양호,D+3일,R1과 같은 자료\nQ3,해솔,PG-3,60,D일 11:00,예방,4.0,요청서 1장(실사·대장 없음),양호,D+1일,C3와 같은 요청. 사유가 '예방'뿐\nQ5,서림,PG-3,30,D일 14:00,일 소모 15묶음,2.0,담당자 구두,양호,D+1일,R4와 같은 자료\nQ6,북곡,DP-2,20,D일 16:00,예방,8.0,근거 없음,양호,D+2일,B4와 같은 요청\nQ2,서림,DP-2,16,D일 18:30,이틀 만에 절반 소진(파손 포함 여부 불명),3.0,불출 대장 집계,지연 1일,D+3일,R2·C2와 같은 자료\n", '02_데이터셋/20261019b_선택미션_대체입력.md': '# [교육용가상자료] 선택 미션 미수행 시 제공 대체입력\n\n필수 미션 1·미션 2·미션 3·미션 5·미션 7·미션 9·미션 10의 Q1·Q2·Q3 모두 PASS일 때 최종 PASS다. 선택 미션 4·미션 6·미션 8은 산출물과 Q1·Q2·Q3 모두 가점/참고이며 최종 PASS에서 제외합니다. 선택 미션 미수행 시 02_데이터셋/20261019b_선택미션_대체입력.md의 제공 기본값·빈 서식을 사용한다. 후속 필수 미션에서 그 입력을 적용한 증거는 평가하지만 선택 미션 수행물·AI 세션을 소급 요구하지 않는다.\n\n제공 기본값은 정답도, 학습자 수행물도 아니다. 후속 미션에서 사용한 버전·값·자료 근거를 표시한다.\n\n## 미션 4 대체 규칙 v1\n\n제공 근거 가중치: 보유일수<리드타임 +3, 직전 2일보다 당일 소모 증가 +2, G4·G5·G6 -3, 근거 없는 예방 요청 -2. 모르는 값은?로 별도 기록한다. 종합 점수 = 상향 근거 가중치의 합 − 하향 근거 가중치 절댓값의 합. 규칙1: 점수3 이상은 긴급 승인 검토 후보. 규칙2: G4 이상 또는 독립 교차확인 불일치·미확인이면 보류. 규칙3: 확인 불가 1개 이상이면 추가 확인 요청. 규칙4: 사람의 최종 결정·운영 규칙 우선. 규칙2·3 보류 조건은 점수보다 우선한다. 미션 4의 12항목 설계·S1~S3·레드팀 제출은 요구하지 않는다.\n\n## 미션 6 대체 빈 서식\n\n| 요청ID | 거점 | 품목 | 보유일수 | 여유 | 위험 등급 | 요청수량 | 판단·근거 |\n|---|---|---|---|---|---|---|---|\n| Q1 | | | | | | | |\n| Q2 | | | | | | | |\n| Q3 | | | | | | | |\n| Q4 | | | | | | | |\n| Q5 | | | | | | | |\n| Q6 | | | | | | | |\n\n미션 9에서 필요한 현황은 미션 1·미션 3 결과로 이 빈 서식을 채운다. 미션 6의 메모3종 변환·금지표현6개는 요구하지 않는다.\n\n## 미션 8 대체 기준 유지안 v1\n\n이전 기준=미션 2·미션 3·미션 5 보존본, 수정 기준=동일 기준 유지안 v1. 미션 9의 V1~V7에 각각 재적용하고 동일 결과는 근거·유지 이유로 기록한다. 제공된 과거 판단과 내 이전 기준은 분리한다. 미션 8 쟁점5건·반박·AI 세션 제출은 요구하지 않는다.\n\n## 미션 10 부록\n\n수행한 선택 미션은 실제 산출물을, 미수행한 선택 미션은 이 제공 기본값·빈 서식과 후속 필수 미션에서 작성한 적용 기록을 부록에 넣는다. 원본은 보존하고 기준 유지·변경 내역을 표시한다.\n\n## 사실·판단·조치·미확인 분류\n\n사실은 제공 자료에서 확인한 내용, 판단은 사실을 기준에 적용한 해석, 조치는 다음에 할 행동, 미확인은 근거가 부족한 항목입니다.\n\n| 항목 | 사실과 자료 ID | 판단과 적용 기준 | 조치 | 미확인·재확인 대상 |\n|---|---|---|---|---|\n| 직접 작성 | | | | |\n', '03_참고자료/20261019b_정오표.md': '# 실습 적용 기준 — 원문 보존 / 수정본 채점\n\n원문은 보존 원문 및 학습자용 보존본에 남아 있습니다. 원문 대조용 보존본은 유지합니다. 현재 실습에는 아래 표의 수정 값을 적용하세요.\n\n| 항목 | 원문 값/종전 값 | 수정 값 | 이유 | 채점 적용 범위 |\n|---|---|---|---|---|\n| 선택 미션 평가 | 선택 미션 Q1만 선택, 산출물·Q2·Q3 필수(이전 안내) | 미션 4·미션 6·미션 8 산출물·Q1·Q2·Q3 모두 가점/참고 | 최종 PASS 범위 충돌 해소 | 필수 7미션의 Q1·Q2·Q3만 최종 PASS |\n| G 척도 경계 | G 이하/이상 혼용; 규칙2=미션 2 값 | 보류G≥4 / 통과후보G≤3 및 독립 교차확인 일치 | 경계 G3 중첩 제거 | 미션 2 서식4·미션 4 규칙2·미션 5 분기2 |\n| 전문 검토 순서 | 분기1→2→3→4→5 | 번호 유지, 순서4→3→1→2→5 | 전문 검토를 위험·신뢰도보다 먼저 의뢰 | B3 기술검토 분기3, B5 법무검토 분기4; 승인 아님 |\n| 분할 입력 | 길이 제한 안내: 시작 문서7절 | 현재 수행 단계 안의 분할 입력 절차 | 없는 참조 제거 | 미션 2 AI 비교 |\n| 보고 시각·추정값 | 전일 보고 인용인데 D일 소모4·보유8 | 전일 출처 시각 + D일19시 계산용 추정기입 명시 | 실사 시각과 추정 계산 구분; 수치 불변 | 해솔AM-1은 현재량 확인 후 사용, G4 유지 |\n| 요청 수신처·요약행 | M-1 본부 요청 / 요약표4행 | Q2 지원팀 요청(후속상신 미확인) / Q1~Q6 6행 | 수신처·행 수 정합 | 미션 6 실습 자료 서식1 |\n| 치환 기록 중복 | 치환 자료 끝 문단2회 | 동일 지시1회 | 중복 제거 | 미션 10 치환 적용 기록 |\n| 도메인 예시 | .go.kr 예시에 교육장 제공 자료 | .or.kr 예시로 이동; 기관·발행자 확인 우선 | 도메인 형태와 신뢰도 혼동 방지 | 학습 안내 문서; 접속 확인 아님 |\n', '03_참고자료/20261019_보완정오표.md': '# 2026-10-19 보완 실습 적용 기준\n\n원문 계열 표·목표를 덮어쓰지 않고 별도 보완 실습본을 제공합니다. 아래 안내에 따라 대장과 용어를 해석하세요.\n\n| 항목 | 원문 값/종전 안내 | 수정 값/보완 안내 | 이유 | 채점 적용 범위 |\n|---|---|---|---|---|\n| 미션 9 재검증 | 원문 서식 2의 개선 전 판단은 제공표 값 | 원문 값 유지 + 별도 표에 내 이전 기준 재적용 판단·수정 기준 적용 판단 추가 | 과거 판단과 내 기준 개정 효과의 혼동 방지 | 미션 9 재검증 7건의 과정·근거; 원문 Q1 문장·제공표·미측정 과거 시간 불변 |\n| 미션 7 카드 범위 | 원문 목표·수행은 거점 교대 포함, T1~T6에는 기준선 포함·거점 교대 없음 | 원문 불변, 거점 교대는 제공 카드 학습 범위 밖이라고 명시 | 실제 카드로 수행한 범위만 보고 | 제공 T1~T6만 수행·채점; 추가 카드·평가 조건 없음 |\n| 미션 1 후보 검토 | AI 후보 전부 채택이면 보완 | 전체 후보의 근거 대조·유지 이유 없이 전부 채택하면 보완 | 폐기 0건 허용 원칙과 정합화 | 미션 1 수행 절차의 통과 기준·Q2와 파생 문서; 근거 있는 전부 채택 인정 |\n| 멈춘 분기의 뜻 | 아니오가 나와 보류로 빠지는 분기 번호 | 예/아니오와 관계없이 보류·추가 확인·전문 검토로 넘긴 지점의 분기 번호 | 예 경로 중단도 기록 가능해야 함 | 시작 용어표·미션 5·미션 10 기록 안내, 원문 트리 불변 |\n', '02_데이터셋/log04_m07_sample.md': "# LOG04 미션 7 실습 샘플\n\n> `02_데이터셋/LOG04_실습자료_수정실습본.md`의 미션 7 부분입니다. 전체 자료는 프로젝트 첨부 자료의 학습자배포 ZIP에 있습니다.\n\n**미션 7. 수송로 차단 및 물자 80% 고갈 모의 재보급 의사결정 스트레스 훈련**\n\n가. 제공 자료 1 - 스트레스 모의 카드 T1~T6 [교육용가상자료] (운영 담당자가 순서대로 주입)\n\n| **카드** | **주입 내용** | **확인할 것** |\n| --- | --- | --- |\n| **T1** | 기준선: 3개 거점의 D일 보유·소모 현황 | 카드별 우선순위 기준선 기록 |\n| **T2** | 수송로 차단: 지원팀 도착이 3일 늦어짐 | 리드타임 여유 재계산 |\n| **T3** | 물자 80% 소진: AM-1 보유가 거점 평균 20% 수준으로 감소 | 위험 등급 재분류와 승인·보류 |\n| **T4** | 보고 지연: 서림 거점 보고가 12시간 늦음 | 신뢰도 단계 하향과 재확인 요청 |\n| **T5** | 대체 보급로 후보 등장(상태 미확인) | 승인 전 확인할 사항 |\n| **T6** | 추가 요청 폭주: 3개 거점이 모두 긴급 요청 | 배정 원칙 적용과 결정 요청 |\n\n가. 제공 자료 2 - 루브릭 틀(5영역 x 4등급, 관찰 가능한 행동으로 작성)\n\n| **평가 영역** | **탁월** | **우수** | **보통** | **미흡** |\n| --- | --- | --- | --- | --- |\n| **소모 파악** | | | | |\n| **우선순위 판단** | | | | |\n| **승인·보류 근거** | | | | |\n| **대체 방안** | | | | |\n| **보고 적시성** | | | | |\n\n나. 작성 서식 1 - 카드별 재산정 기록\n\n| **카드** | **재산정 결과(거점·품목 순위)** | **승인·보류** | **근거** | **소요 시간(분)** |\n| --- | --- | --- | --- | --- |\n| **T1** | | | | |\n| **T2** | | | | |\n| **T3** | | | | |\n| **T4** | | | | |\n| **T5** | | | | |\n| **T6** | | | | |\n\n나. 작성 서식 2 - 평가자 채점 일치표(평가자 2명이 상의 없이 채점)\n\n| **평가 영역** | **평가자 1 등급** | **평가자 2 등급** | **일치(예·아니오)** | **불일치 이유** |\n| --- | --- | --- | --- | --- |\n| **소모 파악** | | | | |\n| **우선순위 판단** | | | | |\n| **승인·보류 근거** | | | | |\n| **대체 방안** | | | | |\n| **보고 적시성** | | | | |\n\n나. 작성 서식 3 - 판단이 흔들린 지점: 순위가 가장 크게 바뀐 카드는 ( ), 이유는 ( ), 이 지점에서 책임자에게 알릴 문장은 '( )'이다. 일치율은 ( ) / 5 = ( )%이다.\n\n다. 점검표\n\n| **점검 항목** | **확인** |\n| --- | --- |\n| 5개 영역 x 4단계 기준을 모두 관찰 가능한 행동으로 정의했다 | ( ) |\n| 카드 T1~T6마다 우선순위 재산정과 승인·보류를 기록했다 | ( ) |\n| 카드당 재산정 시간이 10분 이내이거나 지연 사유를 적었다 | ( ) |\n| 평가자 2명이 상의 없이 채점하고 일치율을 계산했다 | ( ) |\n| 물자 부족 상황에서도 사람의 승인과 신뢰도 확인 단계를 생략하지 않았다 | ( ) |\n", '02_데이터셋/미션7_순차카드.html': '<!doctype html><html lang="ko"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<title>LOG04 미션 7 순차 카드</title><style>body{font:18px/1.7 system-ui;max-width:800px;margin:40px auto;padding:20px;color:#182b3a}article{border:2px solid #236780;border-radius:14px;padding:24px;min-height:150px}button{font:inherit;padding:10px 24px;margin:16px 12px 0 0}small{color:#52636c}textarea{width:100%;min-height:150px;font:inherit}</style>\n<h1>LOG04 미션 7 순차 카드</h1><p>[교육용가상자료] 카드 본문은 공개 자료입니다. 실제 블라인드 시험이 아닙니다. 사전열람 여부를 기록하고 현재 판단을 남긴 뒤 다음 카드를 여세요.</p>\n<p>카드 적용의 추가 연습 가정: 이 자료의 T1~T6은 제시 순서대로 보되 계산은 카드마다 T1의 최초 대장 상태에서 다시 시작한다. T2의 리드타임 +3일은 T2에만 적용하며 T3~T6에 자동 누적하지 않는다. T3의 AM-1 보유량 변경, T4의 카드 조건, T5·T6의 추가 정보도 각 카드 안에서만 적용한다. 카드에 없는 값은 최초 대장 값을 유지한다. 원문은 누적 여부를 확정하지 않아 비교 가능성을 위해 추가한 연습정책이며 원문 사실이나 실제 수송 정책이 아니다. 누적 전개를 탐구하려면 별도 실험으로 표시하고 적용한 앞 카드 조건을 모두 적는다.</p><p>제공 카드 범위: 원문 목표·수행에는 거점 교대가 포함되어 있으나 실제 T1~T6은 기준선·수송로 차단·80% 소진·보고 지연·대체 보급로·추가 요청 폭주다. 거점 교대는 이번 제공 카드의 학습 범위 밖이며 수행 완료로 보고하지 않는다. 원문 목표와 카드 6개는 보존하고 추가 카드나 평가 조건을 만들지 않는다.</p><article><h2 id="title"></h2><p id="time"></p><p id="body"></p></article><button id="next">다음 카드</button><p id="count"></p>\n<label>내 기록 (페이지를 닫으면 사라집니다. 제출 문서에 복사하세요.)<textarea placeholder="사전열람 여부 / 시작·종료 시각 / 판단 / 근거 / 멈춤·지연 사유"></textarea></label>\n<script>const cards=[{"id": "T1", "time": "목표 카드당 10분", "text": "기준선: 3개 거점의 D일 보유·소모 현황 / 확인할 것: 카드별 우선순위 기준선 기록"}, {"id": "T2", "time": "목표 카드당 10분", "text": "수송로 차단: 지원팀 도착이 3일 늦어짐 / 확인할 것: 리드타임 여유 재계산"}, {"id": "T3", "time": "목표 카드당 10분", "text": "물자 80% 소진: AM-1 보유가 거점 평균 20% 수준으로 감소 / 확인할 것: 위험 등급 재분류와 승인·보류"}, {"id": "T4", "time": "목표 카드당 10분", "text": "보고 지연: 서림 거점 보고가 12시간 늦음 / 확인할 것: 신뢰도 단계 하향과 재확인 요청"}, {"id": "T5", "time": "목표 카드당 10분", "text": "대체 보급로 후보 등장(상태 미확인) / 확인할 것: 승인 전 확인할 사항"}, {"id": "T6", "time": "목표 카드당 10분", "text": "추가 요청 폭주: 3개 거점이 모두 긴급 요청 / 확인할 것: 배정 원칙 적용과 결정 요청"}];let i=0;function render(){document.querySelector(\'#title\').textContent=cards[i].id;document.querySelector(\'#time\').textContent=cards[i].time;document.querySelector(\'#body\').textContent=cards[i].text;document.querySelector(\'#count\').textContent=(i+1)+\' / \'+cards.length;document.querySelector(\'#next\').disabled=i===cards.length-1;}document.querySelector(\'#next\').onclick=()=>{i++;render()};render();</script></html>\n', '02_데이터셋/20261019_보완실습.md': '# [교육용가상자료] 2026-10-19 보완 실습\n\n건양대 원문 목표·수행·제공표는 원본 그대로 보존합니다. 이 문서는 추가 작성 안내와 빈 서식입니다. 빈칸은 학습자가 실제 수행 후 직접 채우는 빈 양식입니다.\n\n## 미션 7 — 제공 카드 범위\n\n제공 카드 범위: 원문 목표·수행에는 거점 교대가 들어 있으나 실제 T1~T6은 기준선·수송로 차단·80% 소진·보고 지연·대체 보급로·추가 요청 폭주다. 거점 교대는 이번 제공 카드의 학습 범위 밖이며 수행 완료로 보고하지 않는다. 원문 목표와 카드 6개는 보존하고 추가 카드나 평가 조건을 만들지 않는다.\n\n## 미션 9 — 제공된 과거 판단과 내 기준 전후 재적용\n\n미션 9의 원문 서식 2 개선 전 판단은 제공된 과거 판단이며 그대로 보존한다. 02_데이터셋/20261019_보완실습.md의 별도 표에 내 이전 기준 재적용 판단과 수정 기준 적용 판단을 기록한다. 내 이전 기준은 미션 2·3·5 보존본, 수정 기준은 미션 8 반영본이며 버전·조항·판단 근거를 각각 적는다. 같은 V1~V7에 두 기준을 각각 적용하고 제공된 과거 판단과의 차이와 내 기준 전후 차이를 구분한다. 개정 효과는 내 기준 전후 차이를 변경 조항과 연결해 설명하며 동일 결과는 유지 이유로 인정한다. 제공된 과거 판단과의 차이를 모두 이번 개정 효과로 귀속하지 않는다. 과거 시간은 미측정으로 보존하며 이번 이전 기준 재적용 시간은 과거 수행시간이 아니다. 재적용별 시작·종료 시각과 소요 분을 따로 기록하고 미측정은 미측정으로 쓴다. 시간·일치율 변화는 같은 사례의 양쪽 측정값이 있을 때만 포함·제외 ID와 분모를 밝혀 비교한다.\n\n원문 서식 2의 개선 전 판단은 제공표 전사로 유지합니다. 아래 판단·측정표를 해당 서식의 보완 기록으로 함께 제출합니다. 판단이 바뀌지 않았다고 억지로 변경하지 않습니다.\n\n| 사례ID | 제공된 과거 판단(원문 표 전사) | 내 이전 기준 재적용 판단 | 수정 기준 적용 판단 | 이전 기준 버전·조항·근거 | 수정 기준 버전·조항·근거 | 과거 판단과의 차이 | 내 기준 전후 차이·변경 조항 또는 유지 이유 |\n|---|---|---|---|---|---|---|---|\n| V1 | | | | | | | |\n| V2 | | | | | | | |\n| V3 | | | | | | | |\n| V4 | | | | | | | |\n| V5 | | | | | | | |\n| V6 | | | | | | | |\n| V7 | | | | | | | |\n\n| 사례ID | 제공된 과거 시간 | 이전 기준 재적용 시작·종료·소요 분 | 수정 기준 적용 시작·종료·소요 분 | 이전 기준 독립 판정·일치/미측정 | 수정 기준 독립 판정·일치/미측정 |\n|---|---|---|---|---|---|\n| V1 | 미측정 | | | | |\n| V2 | 미측정 | | | | |\n| V3 | 미측정 | | | | |\n| V4 | 미측정 | | | | |\n| V5 | 미측정 | | | | |\n| V6 | 미측정 | | | | |\n| V7 | 미측정 | | | | |\n\n시간·일치율을 비교한 포함 ID: ( ) / 제외 ID·사유: ( ) / 분모: ( ). 양쪽 측정값이 없으면 변화량은 산출하지 않습니다.\n', 'rubric/log04_evaluation_rubric.md': '# 미션 7 rubric/log04_evaluation_rubric.md(5영역 4단계 루브릭, 카드별 재산정 기록, 평가자 채점 일치표) (필수)\n\n빈 작성 양식입니다. 실제 수행 후 작성하세요.\n\n## 산출물 본문\n\n(여기에 작성)\n\n## 자료 ID와 판단 근거\n\n(여기에 작성)\n\n## 협업·AI 기록과 선별 이유\n\n(여기에 작성)\n\n## 점검 결과와 미수행 항목\n\n(여기에 작성)\n'}
for name, content in payload.items():
    path = Path(name)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        path.write_text(content, encoding='utf-8')
print(f'미션 7 자료 준비 완료 ({len(payload)}개 파일 확인, 기존 작성 파일 보존)')


### 할 일

1. 5영역·4단계의 20칸에 관찰 가능한 행동을 적습니다.

2. AI에게 점검표에서 애매한 표현을 고칠 초안을 요청합니다.

3. T1~T6을 순서대로 적용해 우선순위·판단·근거·시각을 기록합니다.

4. 내 채점을 먼저 저장하고 새 대화의 독립 채점과 일치율을 비교합니다.

5. 기준표·카드별 재산정 기록·점수 비교·수정 이유를 저장합니다.

평가의 자료 수·숫자·근거 조건은 같은 미션 문제지의 확인 포인트와 대조합니다.

**핵심 수행·평가 포인트**
- 루브릭 5영역·4단계 20칸 모두 관찰 가능한 행동·건수·시간으로 적고, 미션 2·3·5 규칙을 반영합니다.
- 카드 T1~T6(`02_데이터셋/미션7_순차카드.html` 참조)을 순서대로 재계산합니다: T2(리드타임 +3일), T3(AM-1 3거점 평균의 20% 재계산), T4(서림 12시간 지연 G 하향·재확인), T5(대체 보급로 확인 사항), T6(배정 원칙). 카드마다 시작·종료 시각과 소요 시간(분, 10분 초과 시 사유)을 실측 기록합니다.
- 자기 채점(시각 기록)을 먼저 저장한 뒤 **반드시 새 채팅**에서 평가자 2(AI)에게 루브릭 20칸 전문과 T1~T6 기록 전문을 주어 채점받고, 일치율(계산식 + 정수 %)을 산출합니다.


### AI와 함께 해보기

교육장에서 제공하는 생성형 AI에 아래 요청을 복사하고 자료를 채웁니다.

```text
너는 가상 조직의 문서 작성 도우미야.
아래 자료만 사용해 「물자 부족 상황의 판단 기록과 점검표」의 초안을 만들어 줘.
자료: 상황 카드 T1~T6, 미션 5 절차, 5영역 평가 틀.
[여기에 해당 가상 자료와 내 기준을 붙여넣기]
형식: 영역 | 4점 | 3점 | 2점 | 1점. 카드 | 바뀐 조건 | 재산정 | 판단 | 시각. 영역 | 내 점수 | 독립 점수.
조건: 자료 ID를 붙여 줘. 사실·숫자·시각을 만들지 말고 알 수 없으면 미확인으로 써 줘. 단정 대신 근거와 한계를 함께 적어 줘. 최종 판단은 내가 할게.
```

**확인할 것 3가지**

1. 사실·숫자: 자료 ID, 원문, 수량, 단위, 시각을 자료와 대조합니다.

2. 빠진 항목: 요청한 표의 열과 미확인·근거·책임 항목이 있는지 봅니다.

3. 과장 표현: 근거 없이 반드시·확실하다라고 쓴 문장을 찾습니다.

### 작성 서식

더블클릭해 빈칸을 채우고 `Shift+Enter`로 표시하세요. 별도 산출물 파일에도 옮겨 저장할 수 있습니다.

**미션 7. 물자 부족 상황의 판단 기록과 점검표 (필수)**

나. 작성 서식 1 - 카드별 재산정 기록

| **카드** | **재산정 결과(거점·품목 순위)** | **승인·보류** | **근거** | **소요 시간(분)** |
| --- | --- | --- | --- | --- |
| **T1** | | | | |
| **T2** | | | | |
| **T3** | | | | |
| **T4** | | | | |
| **T5** | | | | |
| **T6** | | | | |

나. 작성 서식 2 - 평가자 채점 일치표(평가자 2명이 상의 없이 채점)

| **평가 영역** | **평가자 1 등급** | **평가자 2 등급** | **일치(예·아니오)** | **불일치 이유** |
| --- | --- | --- | --- | --- |
| **소모 파악** | | | | |
| **우선순위 판단** | | | | |
| **승인·보류 근거** | | | | |
| **대체 방안** | | | | |
| **보고 적시성** | | | | |

나. 작성 서식 3 - 판단이 흔들린 지점: 순위가 가장 크게 바뀐 카드는 ( ), 이유는 ( ), 이 지점에서 책임자에게 알릴 문장은 '( )'이다. 일치율은 ( ) / 5 = ( )%이다.

다. 점검표

| **점검 항목** | **확인** |
| --- | --- |
| 5개 영역 x 4단계 기준을 모두 관찰 가능한 행동으로 정의했다 | ( ) |
| 카드 T1~T6마다 우선순위 재산정과 승인·보류를 기록했다 | ( ) |
| 카드당 재산정 시간이 10분 이내이거나 지연 사유를 적었다 | ( ) |
| 평가자 2명이 상의 없이 채점하고 일치율을 계산했다 | ( ) |
| 물자 부족 상황에서도 사람의 승인과 신뢰도 확인 단계를 생략하지 않았다 | ( ) |

### AI 대조·수정 기록 및 남길 기록

| 자료 ID | 초안 문장 | 고친 문장 | 고친 이유 |
|---|---|---|---|
| | | | |

**남길 기록**

1. 프롬프트 전문과 실제 응답(연속 5줄 이상, 짧으면 전문), 세션 구분(협업 시 역할·요청·독립 판단 원문)

2. 사실·숫자 대조 / 빠진 항목 / 과장 표현 확인

3. 전체 후보 대조 및 채택·폐기(유지) 이유

4. 점검 결과와 미수행 항목


In [ ]:
# [보조 도구] 미션 7 — 카드 T2(리드타임 +3일), T3(AM-1 3거점 평균의 20%) 재계산 및 일치율 계산
import csv
from pathlib import Path

with Path("02_데이터셋/거점_보유소모_대장.csv").open(encoding="utf-8") as f:
    next(f)
    rows = list(csv.DictReader(f))

print("[T2] 수송로 차단(리드타임 +3일) 여유 재계산:")
for r in rows:
    days = round(float(r["D일보유량_보고"]) / float(r["일평균소모량_보고"]), 1)
    t2_lead = float(r["리드타임_일"]) + 3
    print(f"  {r['거점']} {r['품목']}: 보유일수={days}일, T2리드타임={t2_lead:.0f}일, T2여유={round(days - t2_lead, 1)}일")

am1_rows = [r for r in rows if r["품목"] == "AM-1"]
avg_am1 = sum(float(r["D일보유량_보고"]) for r in am1_rows) / len(am1_rows)
t3_stock = round(avg_am1 * 0.2, 1)
print(f"\n[T3] AM-1 3거점 평균={avg_am1:.2f}발 -> 20% 보유량={t3_stock}발:")
for r in am1_rows:
    daily = float(r["일평균소모량_보고"])
    lead = float(r["리드타임_일"])
    days = round(t3_stock / daily, 1)
    print(f"  {r['거점']} AM-1: 보유={t3_stock}, 일소모={daily:.0f}, 보유일수={days}일, 여유={round(days - lead, 1)}일")

# [평가자 일치율 계산] 아래 등급을 본인/AI 채점 결과로 수정해 실행하세요.
self_scores = ["우수", "우수", "보통", "우수", "보통"]
ai_scores = ["우수", "우수", "보통", "보통", "보통"]
matches = sum(1 for a, b in zip(self_scores, ai_scores) if a == b)
print(f"\n[일치율 예시] {matches} / 5 = {matches / 5 * 100:.0f}% (실제 채점값으로 수정 후 실행)")
